In [17]:
import pandas as pd
import pathlib
import sys
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, "../utils/")
from pinwheels import compute_and_plot_latent_scores, extract_top_pathways_with_cancer, plot_stacked_bar_chart

sys.path.append("../5.drug-dependency")
from utils import load_utils

In [18]:
corum_dir = pathlib.Path("../4.gene-expression-signatures/gsea_results/combined_z_matrix_gsea_results_corum.parquet")
corum_df = pd.read_parquet(corum_dir)

reactome_dir = pathlib.Path("../4.gene-expression-signatures/gsea_results/combined_z_matrix_gsea_results_1.parquet")
reactome_df = pd.read_parquet(reactome_dir)

total_drugs = pd.read_csv("../5.drug-dependency/results/total_drug_df.csv")
merged_df = pd.read_csv("../7.collab-data/results/final_drug_w_paths.csv")

In [19]:
# Drop duplicates to count unique drug appearances per ModelID
unique_drug_model = total_drugs[["ModelID", "name"]].drop_duplicates()

total_modelids = unique_drug_model["ModelID"].nunique()
print(f"Total unique ModelIDs (all cancers): {total_modelids}")

# 1. OVERALL drug appearance frequency
total_modelids = unique_drug_model["ModelID"].nunique()
overall_counts = unique_drug_model["name"].value_counts().reset_index()
overall_counts.columns = ["drug", "count"]
overall_counts["percent"] = overall_counts["count"] / total_modelids * 100

# 2. Subset for brain tumors: Neuroblastoma and Diffuse Glioma
brain_df = total_drugs[total_drugs["OncotreePrimaryDisease"].isin(["Neuroblastoma", "Diffuse Glioma"])]

# Drop duplicates within this subset
brain_unique = brain_df[["ModelID", "name"]].drop_duplicates()

# Count again for brain tumor model IDs
brain_modelids = brain_unique["ModelID"].nunique()
brain_counts = brain_unique["name"].value_counts().reset_index()
brain_counts.columns = ["drug", "count"]
brain_counts["percent"] = brain_counts["count"] / brain_modelids * 100

# Optionally merge both to compare
comparison_df = pd.merge(
    overall_counts,
    brain_counts,
    on="drug",
    how="outer",
    suffixes=("_overall", "_brain_tumors")
).fillna(0)

# Sort by highest percentage in brain tumors
comparison_df = comparison_df.sort_values(by="percent_brain_tumors", ascending=False)

# Save to file if needed
comparison_df.to_csv("drug_appearance_comparison.csv", index=False)

print(comparison_df.head(50))

Total unique ModelIDs (all cancers): 958
                          drug  count_overall  percent_overall  \
117                  SB-225002            550        57.411273   
41                   CCT137690            725        75.678497   
120                  SB-743921            624        65.135699   
155                  alisertib            366        38.204593   
157                  amonafide            305        31.837161   
353                 vinflunine            355        37.056367   
189  carboxypyridine-disulfide            160        16.701461   
46                     CNX-774            393        41.022965   
134                        TMS            278        29.018789   
351                vinblastine            205        21.398747   
102                  PF-562271            379        39.561587   
140                     VE-822            539        56.263048   
86                  NSC-663284            461        48.121086   
249                 idarubicin     

In [20]:
#RNA-seq predicted latent dataframe
latent_dir = pathlib.Path("../7.collab-data/results/phgg_latent_predictions.parquet").resolve()
latent_df = pd.read_parquet(latent_dir)
latent_df['latent_score'] = pd.to_numeric(latent_df['latent_score'], errors='coerce').clip(lower=0)

In [21]:
latent_df.head()

,latent_score,ModelID,model,z,latent_dim_total,init
0,0.233299,radiation,vanillavae,41,200,0
1,0.237382,BT245_SHC202,vanillavae,41,200,0
2,0.188392,DIPG4_SHC202,vanillavae,41,200,0
3,0.284418,DIPG7_SHC202,vanillavae,41,200,0
4,0.227856,DIPG13_SHC202,vanillavae,41,200,0


In [22]:
reactome_dims = pathlib.Path("../5.drug-dependency/results/reactome_paths")
reactome_max = pd.read_parquet(reactome_dims)

corum_dims = pathlib.Path("../5.drug-dependency/results/corum_paths")
corum_max = pd.read_parquet(corum_dims)

drug_dims = pathlib.Path("../5.drug-dependency/results/drug_results")
drug_max = pd.read_parquet(drug_dims)

In [23]:
pathway_merge_df = []
corum_merge_df = []
for sample in latent_df['ModelID'].unique():
    p_df = compute_and_plot_latent_scores(sample, latent_df, reactome_max, "reactome_pathway", "nes_score", "Reactome")
    c_df = compute_and_plot_latent_scores(sample, latent_df, corum_max, "reactome_pathway", "nes_score", "CORUM")
    pathway_merge_df.append(p_df)
    corum_merge_df.append(c_df)

Skipping radiation: merge produced no rows.
Skipping radiation: merge produced no rows.
Skipping BT245_SHC202: merge produced no rows.
Skipping BT245_SHC202: merge produced no rows.
Skipping DIPG4_SHC202: merge produced no rows.
Skipping DIPG4_SHC202: merge produced no rows.
Skipping DIPG7_SHC202: merge produced no rows.
Skipping DIPG7_SHC202: merge produced no rows.
Skipping DIPG13_SHC202: merge produced no rows.
Skipping DIPG13_SHC202: merge produced no rows.
Skipping GBM1_SHC202: merge produced no rows.
Skipping GBM1_SHC202: merge produced no rows.
Skipping 245-1: merge produced no rows.
Skipping 245-1: merge produced no rows.
Skipping 245-2: merge produced no rows.
Skipping 245-2: merge produced no rows.
Skipping D4-1: merge produced no rows.
Skipping D4-1: merge produced no rows.
Skipping D4-2: merge produced no rows.
Skipping D4-2: merge produced no rows.
Skipping D7-1: merge produced no rows.
Skipping D7-1: merge produced no rows.
Skipping D7-2: merge produced no rows.
Skipping 

In [24]:
drug_merge_df = []
for sample in latent_df['ModelID'].unique():
    df = compute_and_plot_latent_scores(sample, latent_df, drug_max, "name", "pearson_correlation", "Drug")
    drug_merge_df.append(df)

radiation
BT245_SHC202
DIPG4_SHC202
DIPG7_SHC202
DIPG13_SHC202
GBM1_SHC202
245-1
245-2
D4-1
D4-2
D7-1
D7-2
G1-1
G1-2
GBM2
DIPG17
GSM7305242
GSM7305243
GSM7305244
GSM7305246
GSM7305247
GSM7305249
GSM7305250
GSM7305251
GSM7305252
GSM7305253
GSM7305254
GSM7305256
GSM7305257
GSM7305258


In [25]:
drug_merge_df = pd.concat(drug_merge_df, ignore_index=True)

In [26]:
drug_merge_df.head()

,latent_score,ModelID,model,z,latent_dim_total,init,name,pearson_correlation,pathway_score
0,0.233299,radiation,vanillavae,41,200,0,cladribine,-0.312672,0.233299
1,0.179099,radiation,vanillavae,40,150,0,axitinib,-0.304600,0.179099
2,0.691408,radiation,betatcvae,4,5,4,3-deazaneplanocin-A,-0.268963,0.691408
3,0.237382,BT245_SHC202,vanillavae,41,200,0,cladribine,-0.312672,0.237382
4,0.258824,BT245_SHC202,vanillavae,40,150,0,axitinib,-0.304600,0.258824


In [27]:
hist_df = pathlib.Path("../5.drug-dependency/results/histogram_results")
drug_hist = pd.read_parquet(hist_df)

In [28]:
drug_merge_df['OncotreePrimaryDisease'] = "Pediatric High-Grade Glioma"
for col in drug_hist.columns:
    if col not in drug_merge_df.columns:
        drug_merge_df[col] = pd.NA  # or np.nan depending on downstream needs

# Reorder columns to match drug_hist
drug_merge_df = drug_merge_df[drug_hist.columns]

# Append the modified dataframe
drug_hist = pd.concat([drug_hist, drug_merge_df], ignore_index=True)

In [29]:
drug_merge_df.head()

,ModelID,model,latent_dim_total,init,seed,OncotreePrimaryDisease,z,latent_score,name,pearson_correlation,pathway_score
0,radiation,vanillavae,200,0,<NA>,Pediatric High-Grade Glioma,41,0.233299,cladribine,-0.312672,0.233299
1,radiation,vanillavae,150,0,<NA>,Pediatric High-Grade Glioma,40,0.179099,axitinib,-0.304600,0.179099
2,radiation,betatcvae,5,4,<NA>,Pediatric High-Grade Glioma,4,0.691408,3-deazaneplanocin-A,-0.268963,0.691408
3,BT245_SHC202,vanillavae,200,0,<NA>,Pediatric High-Grade Glioma,41,0.237382,cladribine,-0.312672,0.237382
4,BT245_SHC202,vanillavae,150,0,<NA>,Pediatric High-Grade Glioma,40,0.258824,axitinib,-0.304600,0.258824


In [30]:
merge_keys = ['z', 'model', 'init', 'latent_dim_total']

In [31]:
correlation_df = pd.read_parquet("../5.drug-dependency/results/drug_correlation.parquet")
correlation_df = correlation_df.rename(columns={'full_model_z': 'latent_dim_total'})

In [32]:
top_drugs_df = (
    drug_merge_df
    .sort_values(["ModelID", "latent_score"], ascending=[True, False])
    .groupby("ModelID")
    .head(200)
    .reset_index(drop=True)
)

In [33]:
merged_df = pd.merge(
    top_drugs_df,
    correlation_df[['z', 'model', 'init', 'latent_dim_total', 'name', 'moa', 'target', 'indication', 'phase', 'Associated Pathways']],
    on=merge_keys + ['name'],
    how='left'
)

In [34]:
merged_df.to_csv("results/final_drug_w_paths.csv")